In [2]:
# %pip install nltk

In [3]:
import re
import time
import requests

import nltk
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline

# LLM pre-training + Data Selection

## Brendan Tomoschuk

#### Download today's lecture, as a notebook or pdf

tomoschuk.github.io/downloads


## Introduction

<div class="sbs">
<div>

<ul>
  <li class="fragment">📊 Senior Data Scientist, Reddit</li>
  <li class="fragment">👨‍🏫 Lecturer, UCSD (HDSI and Psychology Department)</li>
  <li class="fragment">Outside the classroom: my dog & husband, dungeons and dragons 🐉, rock climbing 🧗</li>
</ul>

</div>
<div>
<img src='imgs/intro_picture.jpeg' style="width: 100%;">
</div>
</div>



## What class you're in:

<ul>
  <li class="fragment">This class is a lower division <strong>"Language and Modeling"</strong> course</li>
  <li class="fragment">Class objectives:
    <ul>
      <li>introduce students to both human language and language models</li>
      <li>demystify LLMs</li>
      <li>code in python</li>
      <li>set up upper division classes in cognitive science, linguistics and data science</li>
    </ul>
  </li>
  <li class="fragment">Prerequesites:
    <ul>
      <li>intro python /pandas</li>
      <li>intro statistics</li>
      <li>Not necessarily linear algebra etc.</li>
    </ul>
  </li>
</ul>


## Syllabus

Course description: How do humans learn and use language — and how do large language models do it differently? Through weekly Python labs and discussions grounded in linguistics and cognitive science, students build and evaluate small language models to demystify the technology behind tools like ChatGPT and Claude.

<ul>
  <li class="fragment"><strong>Week 1:</strong>
    <ul>
      <li>What is language? What is a language model?</li>
      <li>Python set up, NLTK package</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 2:</strong>
    <ul>
      <li>Bag of words, TF-IDF</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 3:</strong>
    <ul>
      <li><strong>LLM pre-training</strong> (You are here!)</li>
      <li>Statistical learning</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 4:</strong>
    <ul>
      <li>LLM fine-tuning, LLM alignment</li>
      <li>Supervised vs un-supervised learning</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 5:</strong>
    <ul>
      <li>Syntax and pragmatics</li>
      <li>Self-attention</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 6:</strong>
    <ul>
      <li>Distributional Semantics</li>
      <li>embeddings</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 7:</strong> Model evaluation 1
    <ul>
      <li>Construct validity</li>
      <li>Surprisal & Perplexity</li>
      <li>Theory of mind, other cognitive tests</li>
      <li>LLM benchmarks and leaderboards</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 8:</strong>
    <ul>
      <li>Context windows</li>
      <li>Working memory</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 9:</strong>
    <ul>
      <li>Sociolinguistics</li>
      <li>Data input and bias</li>
    </ul>
  </li>
  <li class="fragment"><strong>Week 10:</strong>
    <ul>
      <li>Open vs closed inputs</li>
      <li>Reasoning models</li>
      <li>Review & prepare for final</li>
    </ul>
  </li>
</ul>


## In class today:

<ul>
  <li class="fragment">Breakdown LLM pre-training:
    <ul>
      <li>Corpus</li>
      <li>Tokenize</li>
      <li>Train</li>
      <li>Evaluate</li>
    </ul>
  </li>
  <li class="fragment">Train our own language model</li>
  <li class="fragment">Discuss how data biases models</li>
  <li class="fragment">Highlight parallels between language models and human language</li>
</ul>


## Have a question? Ask it!

<ul>
  <li class="fragment">I've baked in time for questions and interations.</li>
  <li class="fragment">Prefer anonymity? Ask questions at the <a href="https://docs.google.com/forms/d/e/1FAIpQLScXUqVgmY4CbZHvNOT8JrUaF8ea8XZlsi2XaOcNfSKDsZrzlA/viewform?usp=sharing&ouid=108663971772902393148">google form</a> below and I'll check periodically.</li>
  <li class="fragment">This form also allows me to ask you questions and chat through your answers!</li>
</ul>

<img src="imgs/form.png" width="15%" style="float: right;">


## Why should you learn about LLMs?

<ul>
  <li class="fragment">You want to research cognition?
    <ul>
      <li>LLMs are helping us build a better understanding of human language processing, vision etc.</li>
    </ul>
  </li>
  <li class="fragment">You want to research LLMs?
    <ul>
      <li>Cognitive science is on the cutting edge of how we build and measure these tools</li>
    </ul>
  </li>
  <li class="fragment">You are worried about how LLMs are impacting society?
    <ul>
      <li>Understanding the fundamentals of LLMs help us understand their limitations, inform regulation etc</li>
    </ul>
  </li>
</ul>


## What is a language model?

<div class="sbs">
<div>

<ul>
  <li class="fragment">You know language models like ChatGPT, Claude etc</li>
  <li class="fragment">These are a specific type of large language model (LLM), which is a machine learned model that processes natural language (e.g. English)</li>
  <li class="fragment">Training of these models is typically split into three stages. Today we will focus on <strong>pre-training</strong></li>
</ul>

</div>
<div>
<img src="imgs/training_stages.png" style="width: 100%;">
</div>
</div>


## What is pre-training?

<ul>
  <li class="fragment">LLM learns to <strong>predict</strong> language.</li>
  <li class="fragment"><strong>Self-supervised</strong>
    <ul>
      <li>Model creates its own labels from the data</li>
    </ul>
  </li>
  <li class="fragment">It uses billions or trillions of words (or tokens), and often millions of dollars in compute costs</li>
  <li class="fragment">The price of this stage is what causes LLMs to be driven by for-profit entities</li>
</ul>

<img src="imgs/llm_prediction.jpeg" style="width: 55%; display: block; margin: 0.75em auto 0;">


Today we are going to train our own language model! An N-gram model which is a precusor to the LLM. We'll and talk about how they're similar and different.

## Predictive language models:

<div class="sbs">
<div>

<ul>
  <li class="fragment">Corpus — load in some language data</li>
  <li class="fragment">Tokenize — turn raw data into usable units</li>
  <li class="fragment">Train — build the model</li>
  <li class="fragment">Evaluation — examine the output</li>
</ul>

</div>
<div>
<img src="imgs/training_pipeline.png" style="width: 100%;">
</div>
</div>


## Loading a corpus

<div class="sbs">
<div>

<ul>
  <li class="fragment"><strong><code>corpus</code></strong>: one or more documents containing language. For this class we'll use a collection of public domain books from <a href="https://www.gutenberg.org/">Project Gutenberg</a>, in this case the complete works of Jane Austen.</li>
  <li class="fragment">Let's <strong>Download</strong> plain UTF-8, <strong>strip</strong> the standard Gutenberg header/footer and look at some basic stats about our data before going any further</li>
</ul>

</div>
<div>
<img src="imgs/training_pipeline_corpus.png" style="width: 100%;">
</div>
</div>


In [4]:

# The Complete Project Gutenberg Works of Jane Austen (plain UTF-8)
JANE_AUSTEN_URL = "https://www.gutenberg.org/cache/epub/31100/pg31100.txt"


def get_gutenberg_text(url):
    """Download a Project Gutenberg plain-text file and return only the book body.

    Strip Gutenberg boilerplate using the standard START/END markers, normalize newlines.
    """
    time.sleep(2)
    book_string = requests.get(url, timeout=60).text
    start_pat = r'\*\*\* START OF (?:THIS|THE) PROJECT GUTENBERG EBOOK .+ \*\*\*'
    end_pat = r'\*\*\* END OF (?:THIS|THE) PROJECT GUTENBERG EBOOK'
    s = book_string.replace('\r', '')
    text1 = re.split(start_pat, s, maxsplit=1)[1]
    text = re.split(end_pat, text1, maxsplit=1)[0]
    return text


corpus = get_gutenberg_text(JANE_AUSTEN_URL)
print(f"Corpus length (characters): {len(corpus):,}")


Corpus length (characters): 4,354,160


Need a python reminder? 🙋Ask ChatGPT!🙋

## Let's test our code!



In [11]:
print(corpus[1000:1100])

urned over the almost endless creations
of the last century; and there, if every other leaf were pow


## Tokenizing a corpus

<div class="sbs">
<div>

<ul>
  <li class="fragment"><strong>Tokenize</strong> = turn the string into a <strong>list of tokens</strong>.</li>
  <li class="fragment">Our token list will be made up of:
    <ul>
      <li><strong>words:</strong> pride, and, prejudice</li>
      <li><strong>punctuation:</strong> ! ,</li>
      <li><strong>paragraph breaks:</strong> \x03</li>
    </ul>
  </li>
  <li class="fragment">Let's take a minute to play with <a href="https://platform.openai.com/tokenizer">OpenAI's tokenizer</a></li>
</ul>

</div>
<div>
<img src="imgs/training_pipeline_tokenize.png" style="width: 100%;">
</div>
</div>


In [6]:
def tokenize(book_string):
    """Split text into tokens (words, punctuation) with paragraph START/END markers.

    Paragraph boundaries use ASCII 2 (start) and ASCII 3 (end) as special tokens.
    Punctuation except underscore is split into its own tokens.
    """
    book_string = re.sub(r'\n[\n]+\s*\n*', ' \x03 \x02', book_string).strip('\n')
    book_string = re.sub(r'\s+', ' ', book_string)
    new_string = re.findall(r'([^\w ]{1}|[\w]+)', book_string)
    if new_string[0] != '\x03':
        new_string = ['\x03', '\x02'] + new_string
    if new_string[-1] != '\x02':
        new_string = new_string + ['\x03', '\x02']
    return new_string[1:-1]


tokens = tokenize(corpus)
print(f"Token count: {len(tokens):,}")


Token count: 964,596


Need a python reminder? 🙋Ask ChatGPT!🙋

## Let's test our code!



In [7]:
print(tokens[1000:1020])

['and', 'one', 'remained', 'a', 'widower', ',', 'the', 'other', 'a', 'widow', '.', '\x03', '\x02', 'That', 'Lady', 'Russell', ',', 'of', 'steady', 'age']


## 🧠 Activity question 🧠

<div class="fragment">

When babies need to learn language, do they need to "tokenize"? What would it look like?

</div>

<div class="fragment">

Discuss in groups of 2-4 and send answers to the google form.

</div>

<img src="imgs/form.png" width="15%" style="float: right;">


## Preview: Statistical learning

<ul>
  <li class="fragment">Humans learn words (or "tokens") by extracting patterns from experience
    <ul>
      <li>Continuous speech: unbroken stream of sounds without explicit spaces or labels, and they split words.</li>
      <li>Multiple cues: Real "human tokenization" blends statistics with everything else going on in a brain: vision, tone etc</li>
    </ul>
  </li>
</ul>


## Training

<div class="sbs">
<div>

<ul>
  <li class="fragment">Reminder: conditional probabilities and the chain rule.</li>
  <li class="fragment">Joint probability = multiply <strong>P(next token | earlier tokens)</strong> along a sequence.</li>
</ul>

<div class="fragment">

```py
>>> corpus = 'My name is Brendan. Her name is Jane. Their name is Kai.'
>>> tokens = tokenize(corpus)
>>> tokens
['\x02', 'My', 'name', 'is', 'Brendan', '.', 'Her', 'name', 'is', 'Jane', '.', 'Their', 'name', 'is', 'Kai', '.', '\x03']
```

</div>

</div>
<div>
<img src="imgs/training_pipeline_train.png" style="width: 100%;">
</div>
</div>


## Chain rule on a phrase

<div class="fragment">

$$
\begin{align*}
P(\text{My name is Brendan})
&= P(\text{My}) \cdot P(\text{name | My}) \\
&\quad \cdot P(\text{is | My name}) \cdot P(\text{Brendan | name is}) \\
&\quad \cdot P(\text{. | is Brendan})
\end{align*}
$$

</div>

<div class="fragment">

$$
\begin{align*}
&= \frac{1}{17} \cdot 1 \cdot 1 \\
&\quad \cdot \frac{1}{3} \cdot 1
\end{align*}
$$

</div>

<div class="fragment">

$$
\begin{align*}
&= \frac{1}{51}
\end{align*}
$$

</div>


In [8]:
# N-gram order: 4 means we condition on up to 3 previous tokens.
N = 4

#write training data and a vocab dictionary
train_data, vocab = padded_everygram_pipeline(N, [tokens])
lm = MLE(N)
lm.fit(train_data, vocab)

## Evaluation

<div class="sbs">
<div>

<ul>
  <li class="fragment">Let's start with some basic checks to see how our model did</li>
  <li class="fragment">Later in the class we'll get to metrics for evaluation</li>
</ul>

</div>
<div>
<img src="imgs/training_pipeline_evaluate.png" style="width: 100%;">
</div>
</div>


In [15]:
# Now let's check a phrase we know and fetch the most likely next word
prompt = ["I", "have", "no"]

vocab_tokens = [w for w in lm.vocab if w not in {"<s>", "</s>"}]
prediction = max(vocab_tokens, key=lambda w: lm.score(w, prompt))

print(
    {
        "prompt": prompt,
        "most_likely_next_token": prediction,
        "probability": round(lm.score(prediction, prompt), 3),
    }
)


{'prompt': ['I', 'have', 'no'], 'most_likely_next_token': 'doubt', 'probability': 0.255}


In [21]:
# Top-10 next-token candidates with probabilities for the same prompt.

## generate a dictionary 
vocab_tokens = [w for w in lm.vocab if w not in {"<s>", "</s>"}]

## create a list of tuples containing the words and their probability
scored = [(w, round(lm.score(w, prompt), 3)) for w in vocab_tokens]

## sort by most common
top10_next_tokens = sorted(scored, key=lambda x: x[1], reverse=True)[:10]

print("prompt:", prompt)
for token in top10_next_tokens:
    print(token)

prompt: ['I', 'have', 'no']
('doubt', 0.255)
('idea', 0.082)
('reason', 0.064)
('right', 0.045)
('pleasure', 0.045)
('wish', 0.036)
('notion', 0.036)
('such', 0.027)
('hesitation', 0.027)
('patience', 0.027)


## From counts to guesses

<div class="fragment">

The model is drawing from this as a **probability distribution**, so if the user says "I have no" then the language model will guess that you will say doubt 25.5% of the time.

</div>

<div class="fragment">

<div style="width: min(92vw, 980px); margin: 0.75em auto;">
<table style="width:100%; table-layout:fixed; border-collapse:collapse; margin-top:0.5em;">
<tr>
<td style="width:50%; vertical-align:middle; text-align:center; padding:0 0.35rem;">
<img src="imgs/llm_prediction.jpeg" alt="" style="width:100%; height:auto; display:block; margin:0 auto;">
</td>
<td style="width:50%; vertical-align:middle; text-align:center; padding:0 0.35rem;">
<img src="imgs/gromit.gif" alt="" style="width:100%; height:auto; display:block; margin:0 auto;">
</td>
</tr>
</table>
</div>

</div>


## Self-supervised neural networks

<ul>
  <li class="fragment">Intermediate layers between the input and output apply weighted sums and act like "neurons"</li>
</ul>

<div class="fragment">

<div style="width: min(92vw, 980px); margin: 0.75em auto;">
<img src="imgs/self_supervised.gif" alt="" style="width: 100%; height: auto; display: block;">
</div>

</div>

<div class="fragment">

<p><code>"I"</code> <code>"have"</code> <code>"no"</code> <code>"doubt"</code> <code>"that"</code></p>

</div>



## 🧠 Human Language 🧠

Is human language just...predicition? How do you think human language is similar or different to the predictive model we just built?

- Discuss in groups of 2-4 and send answers to the google form.

<img src="imgs/form.png" width="15%" style="float: right;">


## A few possible answers:

- Similarities: Humans can predict the next word in the sentence and research shows that this is an important part of language development
- Differences: Humans have much more context than 3 words, abstract syntax (how we order words), and many other things we'll talk about in this class!


## We built a language model! How is it different than an LLM?

<div class="fragment">
<table>
<thead>
<tr><th>Step</th><th>N-gram</th><th>LLMs</th></tr>
</thead>
<tbody>
<tr><td><strong>Corpus</strong></td><td>1 million tokens</td><td>10 trillion tokens</td></tr>
<tr><td><strong>Tokenize</strong></td><td>words</td><td>embeddings</td></tr>
<tr><td><strong>Train</strong></td><td>chain-rule</td><td>Deep neural net (Transformer)</td></tr>
<tr><td><strong>Evaluate</strong></td><td>Perplexity</td><td>Perplexity</td></tr>
</tbody>
</table>
</div>


## Preview: Self-attention & Syntax

<ul>
  <li class="fragment"><strong>Self-attention:</strong> nodes <strong>reweights</strong> other positions with context
    <ul>
      <li>context windows</li>
    </ul>
  </li>
  <li class="fragment"><strong>Syntax / pragmatics:</strong> word order and sentence context affect how humans process language
    <ul>
      <li>working memory</li>
    </ul>
  </li>
</ul>


## 🧠 Activity question 🧠

<div class="fragment">

How do we think our Jane Austen dataset impacts our model outputs? What would happen if we used Shakespeare? Reddit? Instagram comments?

</div>

<div class="fragment">

Take a minute to think and send answers to the google form or raise your hand.

</div>

<img src="imgs/form.png" width="15%" style="float: right;">


## Data biases and selection

<ul>
  <li class="fragment">"pleasure" is probably not the 5th most common word to follow "I have no"</li>
  <li class="fragment">Data choices impact our model!</li>
</ul>


## In the news

<div class="sbs">
<div>

<ul>
  <li class="fragment">Recent article about Claude resorting to blackmail if told it will be deleted</li>
  <li class="fragment"><a href="https://www.bbc.com/news/articles/cpqeng9d20go">BBC News — link</a></li>
</ul>

</div>
<div>
<img src="imgs/anthropic.jpeg" style="width: 100%;">
</div>
</div>


## Hallucination example

<ul>
  <li class="fragment">Biases in data can lead to <strong>hallucinations</strong> where LLMs provide you with incorrect information.</li>
</ul>

<div class="fragment">

Lawyer used ChatGPT and ChatGPT made up some legal cases. ChatGPT was predicting words in a reasonable order.
<a href="https://www.nytimes.com/2026/04/21/nyregion/sullivan-cromwell-ai-hallucination.html">NYTimes link</a>

</div>

<div class="fragment">

<p><strong>Other food for thought:</strong></p>
<ul>
  <li>Which languages does the LLM have access to?</li>
  <li>Of all the documents in the corpus, which communities are represented? Which authors? Whose facts?</li>
  <li>Does private data go into the model? What might be the consequences?</li>
</ul>

</div>


## 🧠 Activity question 🧠

<div class="fragment">

Do you think there is bias in human language learning? What would that look like?

</div>

<div class="fragment">

Discuss in groups of 2-4 and send answers to the chat.

</div>

<img src="imgs/form.png" width="15%" style="float: right;">


## Preview: Sociolinguistics

<ul>
  <li class="fragment"><strong>Sounds / phonetics:</strong> Your <strong>accent</strong> is driven by your input data.</li>
  <li class="fragment"><strong>Words and grammar:</strong> Kids learn the <strong>vocabulary</strong> their community uses
    <ul>
      <li>You might say "cheugy" but I honestly don't know how to use it because I don't get any input in that way</li>
    </ul>
  </li>
  <li class="fragment"><strong>Beliefs (politics, religion):</strong> Trusted voices teach <strong>what's true or sacred</strong>; language carries <strong>ideology</strong> and <strong>in-group norms</strong>.</li>
</ul>


## Summary

<ul>
  <li class="fragment">Pre-training is where a language model learns to <strong>predict</strong> language!</li>
  <li class="fragment">N grams are a useful precursor to LLMs.</li>
  <li class="fragment">Human language learning and LLM language learning have lots of parallels.</li>
</ul>


## What's coming up in the class?

<ul>
  <li class="fragment"><strong>Fine-tuning:</strong> turning a language model into a conversational bot or otherwise shaping it to a purpose
    <ul>
      <li>how do we get from language prediction to a helpful chatbot?</li>
    </ul>
  </li>
  <li class="fragment"><strong>Model evaluation:</strong> How do we evaluate whether an LLM model is "good", human like
    <ul>
      <li>surprisal & perplexity</li>
      <li>cognitive metrics</li>
    </ul>
  </li>
</ul>


## Other links

- OpenAI on [why lanuage models hallucinate](https://openai.com/index/why-language-models-hallucinate/)

